In [0]:
 %pip install pymongo==4.10.1

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 40.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pymongo]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

In [0]:

from pymongo import MongoClient
from bson import json_util
import json
from datetime import date
from common_utils.logging import get_logger
from common_utils.ingestor import read_cosmosdb_json, write_raw


# ============================================================
# 0. LOGGER
# ============================================================
logger = get_logger("cosmosdb_ingestion")

# ============================================================
# 1. CONFIG PATH
# ============================================================

dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")


# ============================================================
# 2. READ CONFIG
# ============================================================
with open(config_path, "r") as f:
    config = json.load(f)


# ============================================================
# 3. CONFIG SECTIONS
# ============================================================

source_config = config["source"]
target_config = config["target"]

# ============================================================
# 4. RUN DATE
# ============================================================
run_date = date.today().isoformat() 
logger.info("Fetching todays date %s", run_date)

connection_string = dbutils.secrets.get(scope="retail-platform-dev", key=source_config["connection_string"])

# ============================================================
# 5. READ FROM COSMOSDB SERVER
# ============================================================
df = read_cosmosdb_json(
    spark,
    connection_string,
    source_config["database_name"],
    source_config["collection_name"],
    source_config["data_type"]
)

logger.info("Data successfully read from Cosmos DB")


# ============================================================
# 6. TARGET PATH
# ============================================================
base_path = target_config["base_path"]
folder = target_config["folder"]

target_path = f"{base_path}/{folder}/load_date={run_date}"
logger.info("writiing data to %s", target_path)
# ============================================================
# 7. WRITE RAW
# ============================================================

write_raw(df,target_path,target_config["file_format"],target_config["mode"])
logger.info("Data Succesfully written")


2026-10-28 11:52:22,888 INFO Fetching todays date 2026-10-28
/Workspace/Users/moshiur.mubin0@gmail.com/retaildata_platform/common_utils/ingestor.py:34: UserWarning: You appear to be connected to a CosmosDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/cosmosdb
  client = MongoClient(connection_string)
2026-10-28 11:52:27,337 INFO Data successfully read from Cosmos DB
2026-10-28 11:52:27,338 INFO writiing data to /Volumes/retaildata_platform/bronze/raw_data//cosmosdb_sales_orders/load_date=2026-10-28
2026-10-28 11:52:37,789 INFO Data Succesfully written


In [0]:
import os
print(os.listdir("/Workspace/Users/moshiur.mubin0@gmail.com/retaildata_platform/src/ingestion/config"))

---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
File <command-4913639587847671>, line 2
      1 import os
----> 2 print(os.listdir("/Workspace/Users/moshiur.mubin0@gmail.com/retaildata_platform/src/ingestion/config"))

FileNotFoundError: [Errno 2] No such file or directory: '/Workspace/Users/moshiur.mubin0@gmail.com/retaildata_platform/src/ingestion/config'